# Chronos-2 Benchmarking

| | |
|---|---|
| **Organization** | Amazon Web Services (AWS) |
| **Version** | Chronos-2 (October 2025) |
| **Parameters** | 120M |
| **Input Features** | Target series, optional covariates (past and future known) |

## Key Characteristics

Chronos-2 extends the original Chronos family by introducing support for multivariate and covariate-informed forecasting within a single architecture. The model employs a group attention mechanism that alternates between time attention (aggregating information across patches within a single series) and group attention (aggregating information across all series within a group at each patch index). This design enables in-context learning across related series and covariates without requiring task-specific fine-tuning.

The model was trained on a combination of real-world datasets (GiftEvalPretrain) and large-scale synthetic data generated through multivariate structure imposition on univariate base series. 



---

## 1. Setup & Data Loading

**Validation split:** Train on periods 1–36, validate on 37–42 (identical to Notebook 6)  


**Target:** `Revenue cons. (anon)` at subsegment level  


**Runtime:** T4 GPU (Google Colab) — set manually: Runtime → Change runtime type → T4 GPU


In [1]:
# ── Install ──
!pip install -q "chronos-forecasting>=2.0"

In [2]:
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

print(f'GPU available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Device: {torch.cuda.get_device_name(0)}')

GPU available: True
Device: Tesla T4


In [3]:
# ── Constants ──
PERIOD_COL = 'Anon Period'
TARGET     = 'Revenue cons. (anon)'
SUBSEG_COL = 'TGL Business Subsegment'
VAL_CUTOFF = 36
HORIZON    = 6

# ── Paths ──
from google.colab import drive
drive.mount('/content/drive')               
                                          
DATA_PATH = '/content/drive/MyDrive/Coding/data/features/training_subsegment_fs.parquet'      

# ── Load ──
full_train = pd.read_parquet(DATA_PATH)
train_df   = full_train[full_train[PERIOD_COL] <= VAL_CUTOFF].copy()
val_df     = full_train[full_train[PERIOD_COL] >  VAL_CUTOFF].copy()

print(f'Full train: {full_train.shape}')
print(f'Train (1–36): {train_df.shape} | Val (37–42): {val_df.shape}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Full train: (4237, 110)
Train (1–36): (3524, 110) | Val (37–42): (713, 110)


In [4]:
#   Feature columns are loaded as past covariates for Chronos-2. 
#   (`predict_df` supports them natively)

exclude = [TARGET, PERIOD_COL, SUBSEG_COL,
           'TGL Biz Desc', 'TGL Business Unit',
           'TGL Business Segment', 'Orders cons. (anon)']
feature_cols = [c for c in full_train.columns if c not in exclude]

# ── Subsegments ──
subsegments = sorted(
    train_df.dropna(subset=[TARGET])
    .groupby(SUBSEG_COL, observed=True)
    .size()
    .pipe(lambda s: s[s >= 2].index)
    .tolist()
)

print(f'Feature cols: {len(feature_cols)}')
print(f'Subsegments:  {len(subsegments)}')

Feature cols: 104
Subsegments:  117


In [5]:
# ── Evaluation function ──
def evaluate(results: dict, label: str) -> dict:
    all_true, all_pred = [], []
    for seg, preds in results.items():
        actuals = (val_df[val_df[SUBSEG_COL] == seg]
                   .sort_values(PERIOD_COL)
                   .dropna(subset=[TARGET])[TARGET].values)
        n = min(len(actuals), len(preds))
        all_true.extend(actuals[:n])
        all_pred.extend(preds[:n])
    y, yh = np.array(all_true), np.array(all_pred)
    rmse  = np.sqrt(mean_squared_error(y, yh))
    wmape = np.sum(np.abs(y - yh)) / np.sum(np.abs(y)) * 100
    r2    = r2_score(y, yh)
    print(f'{label} — RMSE: {rmse:,.0f} | wMAPE: {wmape:.1f}% | R²: {r2:.4f}')
    return {'Model': label, 'RMSE': rmse, 'wMAPE': wmape, 'R2': r2}

# ── Results tracker ──
all_metrics = []

---
## 2. Modeling

In [7]:
from chronos import Chronos2Pipeline

chronos2_pipeline = Chronos2Pipeline.from_pretrained(
    'amazon/chronos-2',
    device_map='cuda' if torch.cuda.is_available() else 'cpu'
)
print('Chronos-2 loaded.')

Chronos-2 loaded.


In [8]:
chronos2_results, chronos2_errors = {}, []

for i, seg in enumerate(subsegments):
    train_s = train_df[train_df[SUBSEG_COL] == seg].sort_values(PERIOD_COL).dropna(subset=[TARGET])
    val_s   = val_df[val_df[SUBSEG_COL] == seg].sort_values(PERIOD_COL).dropna(subset=[TARGET])
    if len(train_s) < 4 or len(val_s) == 0:
        continue
    try:
        ctx = train_s[[PERIOD_COL, TARGET] + feature_cols].copy()
        ctx['timestamp'] = pd.date_range(start='2022-01-01', periods=len(ctx), freq='MS')
        ctx = ctx.drop(columns=[PERIOD_COL]).rename(columns={TARGET: 'target'})
        ctx['id'] = seg
        pred_df = chronos2_pipeline.predict_df(
            ctx,
            prediction_length=len(val_s),
            quantile_levels=[0.5],
            id_column='id',
            timestamp_column='timestamp',
            target='target',
        )
        chronos2_results[seg] = pred_df['0.5'].values
    except Exception as e:
        chronos2_errors.append((seg, str(e)))
    if (i + 1) % 25 == 0:
        print(f'{i+1}/{len(subsegments)}')

print(f'Done. Results: {len(chronos2_results)} | Errors: {len(chronos2_errors)}')

25/117
50/117
75/117
100/117
Done. Results: 107 | Errors: 0


In [9]:
metrics_chronos2 = evaluate(chronos2_results, 'Chronos-2')
all_metrics.append(metrics_chronos2)

Chronos-2 — RMSE: 8,139,710 | wMAPE: 9.7% | R²: 0.9858


In [ ]:
# ── Export ──
# rows = [{'Subsegment': seg, 'Period': t+1, 'Chronos2_pred': p}
#         for seg, preds in chronos2_results.items()
#         for t, p in enumerate(preds)]
# pd.DataFrame(rows).to_csv('../data/predictions/chronos2_val_results.csv', index=False)
# print('Saved.')